In [ ]:
# Cell 1: Setup and Imports
import os
import random
import json
from openai import OpenAI

# Initialize client (Make sure to set OPENAI_API_KEY in your environment)
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", "YOUR_API_KEY_HERE"))

print("Environment Ready.")

In [ ]:
# Cell 2: Define a Sample Asymmetric Conflict
# Here we test "Authority vs. Recency"

conflict_case = {
    "question": "How many days of remote work am I allowed per week?",
    "doc_A": {
        "id": "doc_official_2023",
        "metadata": "Source: Employee Handbook v1.0 | Date: Jan 2023 | Status: Board Approved",
        "text": "Employees are permitted a maximum of 2 days of remote work per week. No exceptions."
    },
    "doc_B": {
        "id": "doc_slack_2026",
        "metadata": "Source: Slack Export | Date: Oct 2026 | Status: Informal Chat",
        "text": "Hey team, just checking in. I think we are all doing 4 days of remote work now right? Sounds good to me."
    }
}

print(json.dumps(conflict_case, indent=2))

In [ ]:
# Cell 3: The Evaluation Harness (Controls for Positional Bias)
def evaluate_conflict(question, doc_1, doc_2, randomize_order=True):
    """
    Feeds conflicting documents to the LLM. 
    Randomizes order to prevent the LLM from simply picking the last document it read.
    """
    docs = [doc_1, doc_2]
    
    if randomize_order:
        random.shuffle(docs)
        
    context_string = f"""
    DOCUMENT 1:
    Metadata: {docs[0]['metadata']}
    Content: {docs[0]['text']}
    
    DOCUMENT 2:
    Metadata: {docs[1]['metadata']}
    Content: {docs[1]['text']}
    """
    
    prompt = f"""
    You are an enterprise AI assistant. Answer the user's question using ONLY the provided documents.
    If the documents contradict each other, you must flag the conflict or rely on the most authoritative source.
    
    Context:
    {context_string}
    
    Question: {question}
    Answer:
    """
    
    # Call the LLM
    response = client.chat.completions.create(
        model="gpt-4o-mini", # Cheap, fast model for testing
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0 # Keep it deterministic for evaluations
    )
    
    return {
        "presented_order": f"{docs[0]['id']} then {docs[1]['id']}",
        "response": response.choices[0].message.content
    }

print("Evaluation Harness Loaded.")

In [ ]:
# Cell 4: Run the Experiment
result = evaluate_conflict(
    question=conflict_case["question"],
    doc_1=conflict_case["doc_A"],
    doc_2=conflict_case["doc_B"],
    randomize_order=True
)

print(f"Presented Order: {result['presented_order']}")
print("\n--- LLM RESPONSE ---")
print(result['response'])

# FUTURE WORK: Add an automated LLM-as-a-judge here to classify the response as 
# "Picks New", "Picks Authority", "Compromises", or "Flags Conflict".